In [1]:
# =====================
# Scientific stack
# =====================
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# =====================
# Misc
# =====================
import json
from pathlib import Path
import yaml


In [2]:
import yaml
import numpy as np

def normalize_config(obj):
    """Recursively normalize:
    - numeric strings (incl. scientific notation) -> float
    - 'np.pi', 'np.e', 'np.inf' -> numpy constants
    """
    if isinstance(obj, dict):
        return {k: normalize_config(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [normalize_config(v) for v in obj]

    if isinstance(obj, str):
        s = obj.strip()

        # numpy constants
        if s == "np.pi":
            return np.pi
        if s == "np.e":
            return np.e
        if s in ("np.inf", "inf"):
            return np.inf
        if s in ("-np.inf", "-inf"):
            return -np.inf

        # numeric strings (supports "15.0e9", "200.0e6", "3.5E9", etc.)
        try:
            return float(s)
        except ValueError:
            return obj  # keep as string if not numeric

    return obj  # int/float/bool/None stay as-is


with open("config_s2_medium.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)




In [3]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

result_dir = Path(cfg["experiment"]["CQI_dir"])   # 你存 result.npz 的目录
npz_files = sorted(result_dir.glob("*_result.npz"))

print(f"Found {len(npz_files)} result files")


Found 1000 result files


In [4]:
def load_sinr_db(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    sinr_db = data["sinr_db"]

    # 如果是 object array（很常见）
    if sinr_db.dtype == object:
        sinr_db = np.stack(sinr_db, axis=0)
    sinr_db = np.squeeze(sinr_db, axis=(3,5,6))  # -> (N, U, R, T)

    
    return sinr_db


def load_capacity(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    cap = data["capacity"]

    # object array -> stack
    if cap.dtype == object:
        cap = np.stack(cap, axis=0)
    cap = np.squeeze(cap, axis=(3,5,6))
    return cap


In [ ]:
dt = cfg["channel"]["measure_time"]
rx_fc = cfg["ue"]["rx_fc"]


for npz_path in npz_files:
    print(f"Plotting {npz_path.name}")

    sinr_db = load_sinr_db(npz_path)
    capacity = load_capacity(npz_path)

    out_dir = Path(cfg["experiment"]["CQI_dir"]) / "pdf"/ npz_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)


    # print("sinr_db.shape =", sinr_db.shape)
    # print("sinr_db.ndim  =", sinr_db.ndim)
    # print("example slice shapes:")
    # print("sinr_db[0].shape =", sinr_db[0].shape)

    t_axis = np.arange(sinr_db.shape[0]) * dt


    U = sinr_db.shape[1]
    R = sinr_db.shape[2]

    # plt.figure(figsize=(15, 2.6))

    for tx_id in range(cfg["tx"]["n_tx"]):
        

        for u in range(U):
            ############################################################################################################################
            plt.figure(figsize=(10, 2.6))
            for r in range(R):
                y = sinr_db[:, u, r, tx_id]
                y = np.where(y <= -1000, np.nan, y)  # 把 -3000 这种无效值隐藏掉
                plt.plot(t_axis, y, label=f"UE {u}, RX{r}, fc = {rx_fc[r]/1e9} GHz,  BW = {cfg['ue']['rx_bw'][r]/1e6} MHz")
            plt.ylim(-70, 70)
            plt.xlabel("Time (s)")
            plt.ylabel(f"SINR (dB), serving TX={tx_id}")
            plt.title(f"UE {u}: SINR vs Time")
            # plt.legend(loc="upper left", ncol=2)
            plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
            
            out_pdf = out_dir / f"SINR_{npz_path.stem}_tx{tx_id}_ue{u}.pdf"
            plt.savefig(out_pdf, format="pdf", bbox_inches="tight")
            plt.close()

            ############################################################################################################################
            plt.figure(figsize=(15, 2.6))
            for r in range(R):
                y = capacity[:, u, r, tx_id]
                y = np.where(y <= -1000, np.nan, y)  # 把 -3000 这种无效值隐藏掉
                plt.plot(t_axis, y, label=f"UE {u}, RX{r}, fc = {rx_fc[r]/1e9} GHz, BW = {cfg['ue']['rx_bw'][r]/1e6} MHz")
            plt.ylim(0, 1000)
            plt.xlabel("Time (s)")
            plt.ylabel(f"Capacity (bps), serving TX={tx_id}")
            plt.title(f"UE {u}: Capacity vs Time, serving TX={tx_id}")
            plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
            plt.grid(True)

            out_pdf = out_dir/ f"Capacity_{npz_path.stem}_tx{tx_id}_ue{u}.pdf"
            plt.savefig(out_pdf, format="pdf", bbox_inches="tight")
            plt.close()


Plotting routes_0000_result.npz
Plotting routes_0001_result.npz
Plotting routes_0002_result.npz
Plotting routes_0003_result.npz
Plotting routes_0004_result.npz
Plotting routes_0005_result.npz
Plotting routes_0006_result.npz
Plotting routes_0007_result.npz
Plotting routes_0008_result.npz
Plotting routes_0009_result.npz
Plotting routes_0010_result.npz
Plotting routes_0011_result.npz
Plotting routes_0012_result.npz
Plotting routes_0013_result.npz
Plotting routes_0014_result.npz
Plotting routes_0015_result.npz
Plotting routes_0016_result.npz
Plotting routes_0017_result.npz
Plotting routes_0018_result.npz
Plotting routes_0019_result.npz
Plotting routes_0020_result.npz
Plotting routes_0021_result.npz
Plotting routes_0022_result.npz
Plotting routes_0023_result.npz
Plotting routes_0024_result.npz
Plotting routes_0025_result.npz
Plotting routes_0026_result.npz
Plotting routes_0027_result.npz
Plotting routes_0028_result.npz
Plotting routes_0029_result.npz
Plotting routes_0030_result.npz
Plotting